In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
PJME_data = pd.read_csv(
    "PJME_phase2_preprocessed.csv",
    parse_dates=["Datetime"]
)

In [3]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [4]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


print(train.index.min(), "to", train.index.max())
print(validation.index.min(), "to", validation.index.max())
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)
2002-01-08 01:00:00 to 2013-08-13 16:00:00
2013-08-13 17:00:00 to 2016-02-07 08:00:00
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [5]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [6]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = target_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = target_scaler.transform(
    y_test.values.reshape(-1, 1)
)

print("Scaling completed.")

Scaling completed.


In [7]:
def create_sequences(X, y, input_steps=24, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(
            X[i-input_steps:i]
        )

        y_seq.append(
            y[i:i+output_steps].flatten()
        )

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=24,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=24,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=24,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101609, 24, 15)
y_train: (101609, 24)
X_val: (21737, 24, 15)
y_val: (21737, 24)
X_test: (21737, 24, 15)
y_test: (21737, 24)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [9]:
lstm_model_es = Sequential([
    LSTM(
        64,
        input_shape=(24, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

lstm_model_es.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_model_es.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [11]:
lstm_history_es = lstm_model_es.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.1133 - mae: 0.2372 - val_loss: 0.0867 - val_mae: 0.2120
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0695 - mae: 0.1886 - val_loss: 0.0823 - val_mae: 0.2059
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0643 - mae: 0.1800 - val_loss: 0.0795 - val_mae: 0.2022
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0610 - mae: 0.1745 - val_loss: 0.0786 - val_mae: 0.1992
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - loss: 0.0584 - mae: 0.1703 - val_loss: 0.0782 - val_mae: 0.1999
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0561 - mae: 0.1668 - val_loss: 0.0778 - val_mae: 0.1992
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0543 - mae: 0.1640 - val_loss: 0.0759 - val_mae: 0.1953
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0525 - mae: 0.1614 - val_loss: 0.0791 - val_mae: 0.1995
Epoch 9/15
1588/1588 ━━━━━━━━━━

In [12]:
lstm_model_es_pred_scaled = lstm_model_es.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [13]:
lstm_pred = target_scaler.inverse_transform(
    lstm_model_es_pred_scaled.reshape(-1, 1)
).reshape(lstm_model_es_pred_scaled.shape)

lstm_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [14]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
lstm_mae = mean_absolute_error(
    lstm_actual.flatten(),
    lstm_pred.flatten()
)

lstm_rmse = np.sqrt(
    mean_squared_error(
        lstm_actual.flatten(),
        lstm_pred.flatten()
    )
)

lstm_mape = np.mean(
    np.abs(
        (lstm_actual.flatten() - lstm_pred.flatten())
        / lstm_actual.flatten()
    )
) * 100

lstm_r2 = r2_score(
    lstm_actual.flatten(),
    lstm_pred.flatten()
)

lstm_bias = np.mean(
    lstm_pred.flatten() - lstm_actual.flatten()
)

print("LSTM + Earlystopping")
print("MAE :", lstm_mae)
print("RMSE:", lstm_rmse)
print("MAPE:", lstm_mape)
print("R²  :", lstm_r2)
print("Bias:", lstm_bias)

LSTM + Earlystopping
MAE : 1371.5942926832286
RMSE: 1966.0505445799872
MAPE: 4.308456332861411
R²  : 0.9068668697446858
Bias: -7.358135540154029


In [15]:
print("Epochs completed:", len(lstm_history_es.history["loss"]))

Epochs completed: 13


**Dropout**

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

lstm_dropout_model = Sequential([
    LSTM(
        64,
        input_shape=(24, 15),
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(64, activation="relu"),

    Dropout(0.2),

    Dense(24)
])

lstm_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_dropout_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
lstm_history_dropout = lstm_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 15ms/step - loss: 0.1857 - mae: 0.3227 - val_loss: 0.0968 - val_mae: 0.2319
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.1202 - mae: 0.2625 - val_loss: 0.0874 - val_mae: 0.2196
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1124 - mae: 0.2525 - val_loss: 0.0841 - val_mae: 0.2115
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1092 - mae: 0.2486 - val_loss: 0.0816 - val_mae: 0.2085
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1065 - mae: 0.2451 - val_loss: 0.0817 - val_mae: 0.2103
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.1048 - mae: 0.2431 - val_loss: 0.0793 - val_mae: 0.2084
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.1030 - mae: 0.2408 - val_loss: 0.0846 - val_mae: 0.2148
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1019 - mae: 0.2399 - val_loss: 0.0803 - val_mae: 0.2089
Epoch 9/15
1588/1588 ━━━━━━

In [18]:
dropout_pred_scaled = lstm_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [19]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [20]:
mae = mean_absolute_error(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        dropout_actual.flatten(),
        dropout_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten()
    )
) * 100

r2 = r2_score(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

bias = np.mean(
    dropout_pred.flatten() -
    dropout_actual.flatten()
)

print("== LSTM + DROPOUT ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== LSTM + DROPOUT ==
MAE : 1435.9869157270186
RMSE: 1978.9195155859854
MAPE: 4.598822519549131
R²  : 0.9056436559322364
Bias: 292.5129554768307


**Batch Normalization**

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  BatchNormalization

lstm_bn_model = Sequential([
    LSTM(
        64,
        input_shape=(24, 15)
    ),

    BatchNormalization(),

    Dense(64, activation="relu"),

    BatchNormalization(),

    Dense(24)
])

lstm_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_bn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,712 (104.34 KB)

 Trainable params: 26,456 (103.34 KB)

 Non-trainable params: 256 (1.00 KB)

In [22]:
history_bn = lstm_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1691 - mae: 0.3024 - val_loss: 0.0956 - val_mae: 0.2290
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0959 - mae: 0.2319 - val_loss: 0.0920 - val_mae: 0.2190
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0873 - mae: 0.2206 - val_loss: 0.0833 - val_mae: 0.2054
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0827 - mae: 0.2143 - val_loss: 0.0772 - val_mae: 0.1998
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0791 - mae: 0.2094 - val_loss: 0.0751 - val_mae: 0.1966
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0768 - mae: 0.2063 - val_loss: 0.0794 - val_mae: 0.2010
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0746 - mae: 0.2032 - val_loss: 0.0778 - val_mae: 0.1991
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0728 - mae: 0.2009 - val_loss: 0.0781 - val_mae: 0.1969
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [23]:
bn_pred_scaled = lstm_bn_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [24]:
lstm_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(bn_pred_scaled.shape)

lstm_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(
    lstm_bn_actual.flatten(),
    lstm_bn_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        lstm_bn_actual.flatten(),
        lstm_bn_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (lstm_bn_actual.flatten() - lstm_bn_pred.flatten())
        / lstm_bn_actual.flatten()
    )
) * 100

r2 = r2_score(
    lstm_bn_actual.flatten(),
    lstm_bn_pred.flatten()
)

bias = np.mean(
    lstm_bn_pred.flatten() - lstm_bn_actual.flatten()
)

print("== LSTM + Batch ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== LSTM + Batch ==
MAE : 1497.3504537681347
RMSE: 2152.169810189539
MAPE: 4.745313523054921
R²  : 0.8883990446168706
Bias: 331.9928530416228


**RMSPROP**

In [26]:
from tensorflow.keras.optimizers import RMSprop

lstm_rmsprop = Sequential([
    LSTM(
        64,
        input_shape=(24, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

lstm_rmsprop.compile(
    optimizer=RMSprop(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

lstm_rmsprop.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history_rmsprop = lstm_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1152 - mae: 0.2425 - val_loss: 0.0952 - val_mae: 0.2274
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0721 - mae: 0.1923 - val_loss: 0.0823 - val_mae: 0.2080
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0661 - mae: 0.1828 - val_loss: 0.0807 - val_mae: 0.2048
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0625 - mae: 0.1768 - val_loss: 0.0799 - val_mae: 0.2029
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0599 - mae: 0.1726 - val_loss: 0.0797 - val_mae: 0.2019
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0580 - mae: 0.1693 - val_loss: 0.0761 - val_mae: 0.1943
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0564 - mae: 0.1666 - val_loss: 0.0786 - val_mae: 0.1965
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0551 - mae: 0.1646 - val_loss: 0.0828 - val_mae: 0.2062
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━

In [28]:
rmsprop_pred_scaled = lstm_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [29]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rmsprop_mae = mean_absolute_error(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_rmse = np.sqrt(
    mean_squared_error(
        rmsprop_actual.flatten(),
        rmsprop_pred.flatten()
    )
)

rmsprop_mape = np.mean(
    np.abs(
        (rmsprop_actual.flatten() - rmsprop_pred.flatten())
        / rmsprop_actual.flatten()
    )
) * 100

rmsprop_r2 = r2_score(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_bias = np.mean(
    rmsprop_pred.flatten() -
    rmsprop_actual.flatten()
)

print("=== LSTM + RMSPROP ===")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

=== LSTM + RMSPROP ===
MAE : 1448.469909610875
RMSE: 2081.2529851468817
MAPE: 4.552259525724164
R²  : 0.8956326659487768
Bias: 116.74554798932144


**SGD**

In [31]:
from tensorflow.keras.optimizers import SGD

lstm_sgd = Sequential([
    LSTM(
        64,
        input_shape=(24, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

lstm_sgd.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="mse",
    metrics=["mae"]
)

lstm_sgd.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history_sgd = lstm_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.5893 - mae: 0.5945 - val_loss: 0.3187 - val_mae: 0.4510
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.2298 - mae: 0.3808 - val_loss: 0.1936 - val_mae: 0.3472
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1703 - mae: 0.3254 - val_loss: 0.1656 - val_mae: 0.3175
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1460 - mae: 0.2984 - val_loss: 0.1470 - val_mae: 0.2959
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.1286 - mae: 0.2771 - val_loss: 0.1346 - val_mae: 0.2809
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1166 - mae: 0.2614 - val_loss: 0.1262 - val_mae: 0.2699
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1084 - mae: 0.2500 - val_loss: 0.1194 - val_mae: 0.2600
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1026 - mae: 0.2418 - val_loss: 0.1155 - val_mae: 0.2547
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━

In [33]:
sgd_pred_scaled = lstm_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [34]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sgd_mae = mean_absolute_error(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_rmse = np.sqrt(
    mean_squared_error(
        sgd_actual.flatten(),
        sgd_pred.flatten()
    )
)

sgd_mape = np.mean(
    np.abs(
        (sgd_actual.flatten() - sgd_pred.flatten())
        / sgd_actual.flatten()
    )
) * 100

sgd_r2 = r2_score(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_bias = np.mean(
    sgd_pred.flatten() -
    sgd_actual.flatten()
)
print("=== LSTM + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== LSTM + SGD ===
MAE : 1619.9055719913351
RMSE: 2202.2063143118253
MAPE: 5.131375074640218
R²  : 0.8831494261356626
Bias: 41.03663592316552


Learning Rate Scheduling

In [36]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

lstm_lr = Sequential([
    LSTM(
        64,
        input_shape=(24, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

lstm_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [37]:
history_lr = lstm_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.1116 - mae: 0.2388 - val_loss: 0.0931 - val_mae: 0.2257 - learning_rate: 0.0010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0713 - mae: 0.1911 - val_loss: 0.0894 - val_mae: 0.2155 - learning_rate: 0.0010
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0654 - mae: 0.1814 - val_loss: 0.0770 - val_mae: 0.1981 - learning_rate: 0.0010
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0618 - mae: 0.1756 - val_loss: 0.0809 - val_mae: 0.2045 - learning_rate: 0.0010
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0592 - mae: 0.1714 - val_loss: 0.0758 - val_mae: 0.1971 - learning_rate: 0.0010
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0574 - mae: 0.1684 - val_loss: 0.0825 - val_mae: 0.2067 - learning_rate: 0.0010
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0557 - mae: 0.1661
Epoch 7: ReduceLROnPlateau reducing learning rate

In [38]:
lr_pred_scaled = lstm_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [39]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [40]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr_mae = mean_absolute_error(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_rmse = np.sqrt(
    mean_squared_error(
        lr_actual.flatten(),
        lr_pred.flatten()
    )
)

lr_mape = np.mean(
    np.abs(
        (lr_actual.flatten() - lr_pred.flatten())
        / lr_actual.flatten()
    )
) * 100

lr_r2 = r2_score(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_bias = np.mean(
    lr_pred.flatten() -
    lr_actual.flatten()
)

print("==== LSTM + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== LSTM + RMSPROP + LearningRate ====
MAE : 1410.2147051636646
RMSE: 1998.6879593345095
MAPE: 4.457815486714933
R²  : 0.9037490920798036
Bias: 251.24647654060593


**Layers**

In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_deep = Sequential([

    LSTM(
        64,
        return_sequences=True,
        input_shape=(24, 15)
    ),

    LSTM(
        32,
        return_sequences=False
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

lstm_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 24, 64)         │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,568 (142.84 KB)

 Trainable params: 36,568 (142.84 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
history_deep = lstm_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1230 - mae: 0.2485 - val_loss: 0.0927 - val_mae: 0.2203
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0702 - mae: 0.1911 - val_loss: 0.0841 - val_mae: 0.2126
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0628 - mae: 0.1796 - val_loss: 0.0806 - val_mae: 0.2051
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0579 - mae: 0.1721 - val_loss: 0.0818 - val_mae: 0.2053
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0541 - mae: 0.1664 - val_loss: 0.0774 - val_mae: 0.1976
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0505 - mae: 0.1611 - val_loss: 0.0879 - val_mae: 0.2119
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0471 - mae: 0.1562 - val_loss: 0.0853 - val_mae: 0.2088
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0436 - mae: 0.1511 - val_loss: 0.0866 - val_mae: 0.2091
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [43]:
deep_pred_scaled = lstm_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [44]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

deep_mae = mean_absolute_error(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_rmse = np.sqrt(
    mean_squared_error(
        deep_actual.flatten(),
        deep_pred.flatten()
    )
)

deep_mape = np.mean(
    np.abs(
        (deep_actual.flatten() - deep_pred.flatten())
        / deep_actual.flatten()
    )
) * 100

deep_r2 = r2_score(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_bias = np.mean(
    deep_pred.flatten() -
    deep_actual.flatten()
)

print("==== LSTM + Layers ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

==== LSTM + Layers ==========
MAE : 1519.6695388148364
RMSE: 2214.7553001711126
MAPE: 4.766964569656305
R²  : 0.8818139162795469
Bias: 206.9655315675689


**Neurons**

In [46]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_128 = Sequential([

    LSTM(
        128,
        input_shape=(24, 15)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

lstm_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_128.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_8 (LSTM)                   │ (None, 128)            │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,544 (326.34 KB)

 Trainable params: 83,544 (326.34 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
history_128 = lstm_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1013 - mae: 0.2260 - val_loss: 0.0873 - val_mae: 0.2146
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0661 - mae: 0.1831 - val_loss: 0.0795 - val_mae: 0.2031
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0600 - mae: 0.1731 - val_loss: 0.0806 - val_mae: 0.2017
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0554 - mae: 0.1661 - val_loss: 0.0772 - val_mae: 0.1954
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0514 - mae: 0.1602 - val_loss: 0.0802 - val_mae: 0.1977
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0471 - mae: 0.1540 - val_loss: 0.0838 - val_mae: 0.2012
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0423 - mae: 0.1469 - val_loss: 0.0838 - val_mae: 0.2005
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0378 - mae: 0.1401 - val_loss: 0.0852 - val_mae: 0.2019
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [48]:
lstm128_pred_scaled = lstm_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lstm128_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [49]:
lstm128_pred = target_scaler.inverse_transform(
    lstm128_pred_scaled.reshape(-1, 1)
).reshape(lstm128_pred_scaled.shape)

lstm128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lstm128_mae = mean_absolute_error(
    lstm128_actual.flatten(),
    lstm128_pred.flatten()
)

lstm128_rmse = np.sqrt(
    mean_squared_error(
        lstm_actual.flatten(),
        lstm_pred.flatten()
    )
)

lstm128_mape = np.mean(
    np.abs(
        (lstm128_actual.flatten() - lstm128_pred.flatten())
        / lstm128_actual.flatten()
    )
) * 100

lstm128_r2 = r2_score(
    lstm128_actual.flatten(),
    lstm128_pred.flatten()
)

lstm128_bias = np.mean(
    lstm128_pred.flatten() -
    lstm128_actual.flatten()
)

print("=== LSTM 128 NEURONS ===")
print("MAE :", lstm128_mae)
print("RMSE:", lstm128_rmse)
print("MAPE:", lstm128_mape)
print("R²  :", lstm128_r2)
print("Bias:", lstm128_bias)

=== LSTM 128 NEURONS ===
MAE : 1506.9126826623833
RMSE: 1966.0505445799872
MAPE: 4.712792519674344
R²  : 0.8858572879605617
Bias: 126.27973308856599


32 **neurons**

In [51]:
lstm_32 = Sequential([
    LSTM(
        32,
        input_shape=(24, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

lstm_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_32.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_9 (LSTM)                   │ (None, 32)             │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,816 (38.34 KB)

 Trainable params: 9,816 (38.34 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
history_32 = lstm_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1406 - mae: 0.2666 - val_loss: 0.0930 - val_mae: 0.2244
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0744 - mae: 0.1970 - val_loss: 0.0852 - val_mae: 0.2100
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0686 - mae: 0.1872 - val_loss: 0.0826 - val_mae: 0.2064
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0654 - mae: 0.1817 - val_loss: 0.0780 - val_mae: 0.2000
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0631 - mae: 0.1778 - val_loss: 0.0801 - val_mae: 0.2044
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0615 - mae: 0.1752 - val_loss: 0.0747 - val_mae: 0.1942
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0601 - mae: 0.1729 - val_loss: 0.0786 - val_mae: 0.2033
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0588 - mae: 0.1707 - val_loss: 0.0768 - val_mae: 0.1976
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [53]:
lstm32_pred_scaled = lstm_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lstm32_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [54]:
lstm32_pred = target_scaler.inverse_transform(
    lstm32_pred_scaled.reshape(-1, 1)
).reshape(lstm32_pred_scaled.shape)

lstm32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [55]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lstm32_mae = mean_absolute_error(
    lstm32_actual.flatten(),
    lstm32_pred.flatten()
)

lstm32_rmse = np.sqrt(
    mean_squared_error(
        lstm32_actual.flatten(),
        lstm32_pred.flatten()
    )
)

lstm32_mape = np.mean(
    np.abs(
        (lstm32_actual.flatten() - lstm32_pred.flatten())
        / lstm32_actual.flatten()
    )
) * 100

lstm32_r2 = r2_score(
    lstm32_actual.flatten(),
    lstm32_pred.flatten()
)

lstm32_bias = np.mean(
    lstm32_pred.flatten() -
    lstm32_actual.flatten()
)

print("=== LSTM 32 NEURONS ===")
print("MAE :", lstm32_mae)
print("RMSE:", lstm32_rmse)
print("MAPE:", lstm32_mape)
print("R²  :", lstm32_r2)
print("Bias:", lstm32_bias)

=== LSTM 32 NEURONS ===
MAE : 1464.3484043174512
RMSE: 2061.1117565050754
MAPE: 4.648727127650058
R²  : 0.8976429116340158
Bias: 341.37884747145324


Batch Size(32)

In [56]:
lstm_batch32 = Sequential([
    LSTM(64, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = lstm_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.1005 - mae: 0.2246 - val_loss: 0.0841 - val_mae: 0.2091
Epoch 2/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.0682 - mae: 0.1856 - val_loss: 0.0785 - val_mae: 0.1982
Epoch 3/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0625 - mae: 0.1763 - val_loss: 0.0778 - val_mae: 0.1990
Epoch 4/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.0588 - mae: 0.1704 - val_loss: 0.0791 - val_mae: 0.2010
Epoch 5/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0561 - mae: 0.1662 - val_loss: 0.0768 - val_mae: 0.1951
Epoch 6/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0537 - mae: 0.1627 - val_loss: 0.0742 - val_mae: 0.1912
Epoch 7/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - loss: 0.0512 - mae: 0.1592 - val_loss: 0.0807 - val_mae: 0.2048
Epoch 8/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0489 - mae: 0.1562 - val_loss: 0.0809 - val_mae: 0.1992
Epoch 9/15
3176/3176 ━━━━━━━━━━━

In [57]:
lstm_batch32_pred_scaled = lstm_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", lstm_batch32_pred_scaled.shape)

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Prediction shape: (21737, 24)


In [58]:
lstmbatch32_pred = target_scaler.inverse_transform(
    lstm_batch32_pred_scaled.reshape(-1, 1)
).reshape(lstm_batch32_pred_scaled.shape)

lstmbatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [59]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lstmb32_mae = mean_absolute_error(
    lstmbatch32_actual.flatten(),
    lstm32_pred.flatten()
)

lstmb32_rmse = np.sqrt(
    mean_squared_error(
        lstmbatch32_actual.flatten(),
        lstmbatch32_pred.flatten()
    )
)

lstmb32_mape = np.mean(
    np.abs(
        (lstmbatch32_actual.flatten() - lstmbatch32_pred.flatten())
        / lstmbatch32_actual.flatten()
    )
) * 100

lstmb32_r2 = r2_score(
    lstmbatch32_actual.flatten(),
    lstmbatch32_pred.flatten()
)

lstmb32_bias = np.mean(
    lstmbatch32_pred.flatten() -
    lstmbatch32_actual.flatten()
)

print("=== LSTM batch32 NEURONS ===")
print("MAE :", lstmb32_mae)
print("RMSE:", lstmb32_rmse)
print("MAPE:", lstmb32_mape)
print("R²  :", lstmb32_r2)
print("Bias:", lstmb32_bias)

=== LSTM batch32 NEURONS ===
MAE : 1464.3484043174512
RMSE: 2154.173550570659
MAPE: 4.614184993803667
R²  : 0.8881911396102412
Bias: 183.53409682098064


BatchSize(128)

In [60]:
lstm_batch128 = Sequential([
    LSTM(64, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = lstm_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.1442 - mae: 0.2658 - val_loss: 0.0899 - val_mae: 0.2184
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0724 - mae: 0.1935 - val_loss: 0.0838 - val_mae: 0.2099
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.0665 - mae: 0.1837 - val_loss: 0.0812 - val_mae: 0.2041
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0627 - mae: 0.1773 - val_loss: 0.0789 - val_mae: 0.2010
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0601 - mae: 0.1732 - val_loss: 0.0775 - val_mae: 0.1967
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0581 - mae: 0.1699 - val_loss: 0.0755 - val_mae: 0.1939
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0564 - mae: 0.1672 - val_loss: 0.0762 - val_mae: 0.1962
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.0549 - mae: 0.1649 - val_loss: 0.0756 - val_mae: 0.1942
Epoch 9/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - lo

In [61]:
lstm_batch128_pred_scaled = lstm_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", lstm_batch128_pred_scaled.shape)

170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [62]:
lstmbatch128_pred = target_scaler.inverse_transform(
    lstm_batch128_pred_scaled.reshape(-1, 1)
).reshape(lstm_batch128_pred_scaled.shape)

lstmbatch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [63]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lstmb128_mae = mean_absolute_error(
    lstmbatch128_actual.flatten(),
    lstmbatch128_pred.flatten()
)

lstmb128_rmse = np.sqrt(
    mean_squared_error(
        lstmbatch128_actual.flatten(),
        lstmbatch128_pred.flatten()
    )
)

lstmb128_mape = np.mean(
    np.abs(
        (lstmbatch128_actual.flatten() - lstmbatch128_pred.flatten())
        / lstmbatch128_actual.flatten()
    )
) * 100

lstmb128_r2 = r2_score(
    lstmbatch128_actual.flatten(),
    lstmbatch128_pred.flatten()
)

lstmb128_bias = np.mean(
    lstmbatch128_pred.flatten() -
    lstmbatch128_actual.flatten()
)

print("=== LSTM batch128  ===")
print("MAE :", lstmb128_mae)
print("RMSE:", lstmb128_rmse)
print("MAPE:", lstmb128_mape)
print("R²  :", lstmb128_r2)
print("Bias:", lstmb128_bias)

=== LSTM batch128  ===
MAE : 1473.614765763972
RMSE: 2096.804859931053
MAPE: 4.670255980671957
R²  : 0.8940670975664853
Bias: 382.6896847798636


**HyperParameter Tuning**

In [66]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.3 MB/s eta 0:00:00


In [67]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [68]:
def build_lstm(hp):

    model = Sequential()

    model.add(
        LSTM(
            units=hp.Choice(
                "rnn_units",
                values=[32, 64, 128]
            ),
            input_shape=(24, 15)
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [69]:
tuner = kt.RandomSearch(
    build_lstm,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="lstm_48_to_24"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [70]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 02m 44s]
val_loss: 0.07687690109014511

Best val_loss So Far: 0.07353559136390686
Total elapsed time: 00h 18m 06s


In [71]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:",best_hp.get("rnn_units"))
print("Dense Units:",best_hp.get("dense_units"))
print("Learning Rate:",best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 32
Dense Units: 128
Learning Rate: 0.0005


In [72]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,464 (52.59 KB)

 Trainable params: 13,464 (52.59 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0579 - mae: 0.1691 - val_loss: 0.0795 - val_mae: 0.2005
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0572 - mae: 0.1678 - val_loss: 0.0751 - val_mae: 0.1957
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0565 - mae: 0.1666 - val_loss: 0.0806 - val_mae: 0.2019
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0561 - mae: 0.1659 - val_loss: 0.0751 - val_mae: 0.1956
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0554 - mae: 0.1649 - val_loss: 0.0795 - val_mae: 0.1991
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0549 - mae: 0.1641 - val_loss: 0.0768 - val_mae: 0.1972
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0548 - mae: 0.1638 - val_loss: 0.0772 - val_mae: 0.1965
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0541 - mae: 0.1629 - val_loss: 0.0733 - val_mae: 0.1918
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [74]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [75]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

best_mae = mean_absolute_error(
    best_actual.flatten(),
    best_pred.flatten()
)

best_rmse = np.sqrt(
    mean_squared_error(
        best_actual.flatten(),
        best_pred.flatten()
    )
)

best_mape = np.mean(
    np.abs(
        (best_actual.flatten() - best_pred.flatten())
        / best_actual.flatten()
    )
) * 100

best_r2 = r2_score(
    best_actual.flatten(),
    best_pred.flatten()
)

best_bias = np.mean(
    best_pred.flatten() -
    best_actual.flatten()
)

print("HYPERPARAMETER TUNED LSTM")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED LSTM
MAE : 1429.862873023633
RMSE: 2002.5690622350573
MAPE: 4.521391137043684
R²  : 0.9033749242463657
Bias: 243.80461904358424


**Additional Logs**

In [76]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [77]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144888, 21)


In [78]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [79]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101421, 21)
Validation: (21733, 21)
Test: (21734, 21)


In [80]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [81]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [82]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=24,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=24,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=24,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101374, 24, 21)
y_train: (101374, 24)
X_val: (21686, 24, 21)
y_val: (21686, 24)
X_test: (21687, 24, 21)
y_test: (21687, 24)


In [83]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_lags = Sequential([
    LSTM(
        64,
        input_shape=(24, 21)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

lstm_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_lags.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        22,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,736 (108.34 KB)

 Trainable params: 27,736 (108.34 KB)

 Non-trainable params: 0 (0.00 B)

In [84]:
history_lags = lstm_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1131 - mae: 0.2383 - val_loss: 0.0859 - val_mae: 0.2113
Epoch 2/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0686 - mae: 0.1874 - val_loss: 0.0801 - val_mae: 0.2063
Epoch 3/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0630 - mae: 0.1780 - val_loss: 0.0775 - val_mae: 0.2002
Epoch 4/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0596 - mae: 0.1725 - val_loss: 0.0777 - val_mae: 0.1987
Epoch 5/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0569 - mae: 0.1682 - val_loss: 0.0738 - val_mae: 0.1939
Epoch 6/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0548 - mae: 0.1650 - val_loss: 0.0780 - val_mae: 0.1997
Epoch 7/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0524 - mae: 0.1616 - val_loss: 0.0755 - val_mae: 0.1979
Epoch 8/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0501 - mae: 0.1583 - val_loss: 0.0759 - val_mae: 0.1925
Epoch 9/15
1584/1584 ━━━━━━━━━━━

In [85]:
pred_scaled = lstm_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

339/339 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21687, 24)


In [86]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [87]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(
    actual.flatten(),
    pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        actual.flatten(),
        pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (actual.flatten() - pred.flatten())
        / actual.flatten()
    )
) * 100

r2 = r2_score(
    actual.flatten(),
    pred.flatten()
)

bias = np.mean(
    pred.flatten() - actual.flatten()
)

print("=== LSTM + ADDITIONAL LAGS ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== LSTM + ADDITIONAL LAGS ===
MAE : 1468.0982204703507
RMSE: 2120.8439884201957
MAPE: 4.624524339084943
R²  : 0.8917507422669871
Bias: 269.7069927232052


In [2]:
import pandas as pd
import os

lstm_24_24_results = [
    ["LSTM Baseline", 24, 24, 1491.6327254665684, 2165.564133063436, 4.693110195238567, 0.8870055943155081, 315.4325268070607],
    ["LSTM + EarlyStopping", 24, 24, 1371.5942926832286, 1966.0505445799872, 4.308456332861411, 0.9068668697446858, -7.358135540154029],
    ["LSTM + Dropout", 24, 24, 1435.9869157270186, 1978.9195155859854, 4.598822519549131, 0.9056436559322364, 292.5129554768307],
    ["LSTM + Batch Normalization", 24, 24, 1497.3504537681347, 2152.169810189539, 4.745313523054921, 0.8883990446168706, 331.9928530416228],
    ["LSTM + RMSprop", 24, 24, 1448.469909610875, 2081.2529851468817, 4.552259525724164, 0.8956326659487768, 116.74554798932144],
    ["LSTM + SGD", 24, 24, 1619.9055719913351, 2202.2063143118253, 5.131375074640218, 0.8831494261356626, 41.03663592316552],
    ["LSTM + RMSprop + Learning Rate", 24, 24, 1410.2147051636646, 1998.6879593345095, 4.457815486714933, 0.9037490920798036, 251.24647654060593],
    ["LSTM + Additional Layers", 24, 24, 1519.6695388148364, 2214.7553001711126, 4.766964569656305, 0.8818139162795469, 206.9655315675689],
    ["LSTM 128 Neurons", 24, 24, 1506.9126826623833, 1966.0505445799872, 4.712792519674344, 0.8858572879605617, 126.27973308856599],
    ["LSTM 32 Neurons", 24, 24, 1464.3484043174512, 2061.1117565050754, 4.648727127650058, 0.8976429116340158, 341.37884747145324],
    ["LSTM Batch Size 32", 24, 24, 1464.3484043174512, 2154.173550570659, 4.614184993803667, 0.8881911396102412, 183.53409682098064],
    ["LSTM Batch Size 128", 24, 24, 1473.614765763972, 2096.804859931053, 4.670255980671957, 0.8940670975664853, 382.6896847798636],
    ["LSTM Hyperparameter Tuned", 24, 24, 1429.862873023633, 2002.5690622350573, 4.521391137043684, 0.9033749242463657, 243.80461904358424],
    ["LSTM + Additional Lags", 24, 24, 1468.0982204703507, 2120.8439884201957, 4.624524339084943, 0.8917507422669871, 269.7069927232052]
]

lstm_24_24_df = pd.DataFrame(
    lstm_24_24_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

# Display with exactly 6 decimal places
display(
    lstm_24_24_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

# Save inside existing comparison folder
os.makedirs("comparison", exist_ok=True)

lstm_24_24_df.to_csv(
    "comparison/LSTM_24h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved: comparison/LSTM_24h_to_24h_results.csv")


,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,LSTM Baseline,24,24,1491.632725,2165.564133,4.693110,0.887006,315.432527
1,LSTM + EarlyStopping,24,24,1371.594293,1966.050545,4.308456,0.906867,-7.358136
2,LSTM + Dropout,24,24,1435.986916,1978.919516,4.598823,0.905644,292.512955
3,LSTM + Batch Normalization,24,24,1497.350454,2152.169810,4.745314,0.888399,331.992853
4,LSTM + RMSprop,24,24,1448.469910,2081.252985,4.552260,0.895633,116.745548
5,LSTM + SGD,24,24,1619.905572,2202.206314,5.131375,0.883149,41.036636
6,LSTM + RMSprop + Learning Rate,24,24,1410.214705,1998.687959,4.457815,0.903749,251.246477
7,LSTM + Additional Layers,24,24,1519.669539,2214.755300,4.766965,0.881814,206.965532
8,LSTM 128 Neurons,24,24,1506.912683,1966.050545,4.712793,0.885857,126.279733
9,LSTM 32 Neurons,24,24,1464.348404,2061.111757,4.648727,0.897643,341.378847


Saved: comparison/LSTM_24h_to_24h_results.csv


In [3]:
best_lstm_24_24 = lstm_24_24_df.loc[lstm_24_24_df["Model"] == "LSTM + EarlyStopping"]
display(best_lstm_24_24.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
1,LSTM + EarlyStopping,24,24,1371.594293,1966.050545,4.308456,0.906867,-7.358136
